# 목표

번역기 만들기

In [51]:
import os

ROOT_DIR = os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, "data")

train_json_path = os.path.join(DATA_DIR, "raw", "일상생활및구어체_한영_train_set.json")
val_json_path = os.path.join(DATA_DIR, "raw", "일상생활및구어체_한영_valid_set.json")

In [52]:
import json

with open(train_json_path, "r", encoding="utf-8") as f:
    TRAIN_DATA = json.load(f)

TRAIN_DATA = TRAIN_DATA["data"]

In [53]:
import random

rand_idx = random.randint(1, len(TRAIN_DATA)) - 1

TRAIN_DATA[26]

{'sn': 'KPUS062012215029114',
 'data_set': '일상생활및구어체',
 'domain': '일상생활',
 'subdomain': '구매',
 'ko_original': '>와우영미.',
 'ko': '>와우영미.',
 'mt': '> Wow, Woo Youngmi.',
 'en': '> WaWooYoungmi.',
 'source_language': 'ko',
 'target_language': 'en',
 'word_count_ko': 1,
 'word_count_en': 2,
 'word_ratio': 2.0,
 'file_name': '구매_KPUS.xlsx',
 'source': 'SBS',
 'license': 'open',
 'style': '구어체',
 'included_unknown_words': False,
 'ner': {'text': '>와우<PERSON>영미</PERSON>.',
  'tags': [{'tag': 'PERSON', 'value': '영미', 'position': '[3, 5]'}]}}

ner은 Named Entity Recognition. 꼭 살려둬야 한다.

In [54]:
print("[데이터 속성 고유값 확인]")

unique_dict = {key: (set() if key != "ner" else list()) for key in TRAIN_DATA[0].keys()}


for i, dict_ in enumerate(TRAIN_DATA):   
    for key, value in dict_.items():
        if key == "ner":
            unique_dict[key].append(value if value else "null")
        else:
            unique_dict[key].add(value if value else "null")


for key, value in unique_dict.items():
    print(f"· {key}: {len(value)}개")

    if len(value) < 20:
        print(f"    - {value}")

[데이터 속성 고유값 확인]
· sn: 1200000개
· data_set: 1개
    - {'일상생활및구어체'}
· domain: 3개
    - {'일상생활', '해외고객과의채팅', '해외영업'}
· subdomain: 11개
    - {'금융,보험', '숙박,음식점', '정보통신', '기계장비,의료정밀', '연구개발,과학기술', '여행', '음식', '예약', '도소매유통', '구매', '부동산'}
· ko_original: 1059799개
· ko: 1059798개
· mt: 1015045개
· en: 1024329개
· source_language: 1개
    - {'ko'}
· target_language: 1개
    - {'en'}
· word_count_ko: 56개
· word_count_en: 73개
· word_ratio: 673개
· file_name: 18개
    - {'INTSAL_MCHO.xlsx', 'CUSCHA_STRS.xlsx', 'CUSCHA_DSUT.xlsx', 'CUSCHA_EGKG.xlsx', 'CUSCHA_RLST.xlsx', '예약_KRSS.xlsx', 'CUSCHA_MCHO.xlsx', 'CUSCHA_FNIN.xlsx', 'INTSAL_STRS.xlsx', 'INTSAL_RLST.xlsx', 'INTSAL_DSUT.xlsx', '구매_KPUS.xlsx', 'INTSAL_JBTS.xlsx', 'CUSCHA_JBTS.xlsx', 'INTSAL_EGKG.xlsx', '음식_KFDS.xlsx', '여행_KTOS.xlsx', 'INTSAL_FNIN.xlsx'}
· source: 2개
    - {'크라우드소싱', 'SBS'}
· license: 1개
    - {'open'}
· style: 1개
    - {'구어체'}
· included_unknown_words: 1개
    - {'null'}
· ner: 1200000개


In [ ]:
useless_list = [
    "sn", "data_set", "license", "source", "style", 
    "included_unknown_words", "file_name", 
    "source_language", "target_language"
    ]

for dict_ in TRAIN_DATA:   
    for trash in useless_list:
        del dict_[trash]

내용 중복들도 있나보다. 숫자가 적은 거 보니까.
지우고 가야지.

In [56]:
import pandas as pd

train_df = pd.DataFrame(TRAIN_DATA)

In [62]:
dup_list = ["ko_original", "ko", "mt", "en"]
train_df.loc[train_df.duplicated(dup_list)].head(5)

for column in dup_list:
    train_df = train_df.drop_duplicates(["domain", "subdomain", column], keep="first")

print(f"· 중복 제거 후 데이터: {len(train_df)}개")

· 중복 제거 후 데이터: 1024631개


,sn,domain,subdomain,ko_original,ko,mt,en,word_count_ko,word_count_en,word_ratio,ner
0,INTSALDSUT062119042703238,해외영업,도소매유통,원하시는 색상을 회신해 주시면 바로 제작 들어가겠습니다.,원하시는 색상을 회신해 주시면 바로 제작 들어가겠습니다.,"If you reply to the color you want, we will st...","If you reply to the color you want, we will st...",7,15,2.143,None
1,KTOS062012215152657,일상생활,여행,형님 제일 웃긴 그림이 뭔지 알아요.,형님 제일 웃긴 그림이 뭔지 알아요.,I know what the funniest picture is.,You know what the funniest picture is.,6,7,1.167,None
2,KRSS062012215033840,일상생활,예약,>속옷을?,>속옷을?,Underwear?,>Underwear?,1,1,1.000,None
3,INTSALEGKG062119042674878,해외영업,"연구개발,과학기술",그래도 가격이 꽤 비싸니까 많이 살게요.,그래도 가격이 꽤 비싸니까 많이 살게요.,"However, the price is quite high, so I will bu...",I wont buy a lot though since the price is sti...,6,13,2.167,None
4,CUSCHADSUT062119042866224,해외고객과의채팅,도소매유통,"AAA님, 제가 회의에서 화를 냈던 점 정말 사과드리고 싶습니다.","AAA님, 제가 회의에서 화를 냈던 점 정말 사과드리고 싶습니다.","AAA, I really want to apologize for being upse...","Dear AAA, I really want to apologize for my an...",9,13,1.444,None
...,...,...,...,...,...,...,...,...,...,...,...
1199994,INTSALDSUT0621190427217437,해외영업,도소매유통,당사와 귀사는 오랜 기간 협력 관계였기 때문에 되도록이면 귀하의 요청을 들어드리고 ...,당사와 귀사는 오랜 기간 협력 관계였기 때문에 되도록이면 귀하의 요청을 들어드리고 ...,Since we and your company have been working to...,Since we have been in a cooperative relationsh...,12,23,1.917,None
1199995,INTSALEGKG062119042669888,해외영업,"연구개발,과학기술",그럼 미디엄은 2리터 필요하신 거죠?,그럼 미디엄은 2리터 필요하신 거죠?,So you need 2 liters of medium?,So then you need 2 liters of the medium right?,5,10,2.000,None
1199996,CUSCHADSUT0621190428113186,해외고객과의채팅,도소매유통,환불 요청은 BBB 고객 지원으로 문의하십시오.,환불 요청은 BBB 고객 지원으로 문의하십시오.,Please contact BBB customer support for refund...,Please contact BBB customer support for refund...,6,8,1.333,None
1199997,INTSALDSUT0621190427147706,해외영업,도소매유통,미니 사이즈로 언제어디서나 즐길 수 있는 휴대성을 자랑합니다.,미니 사이즈로 언제어디서나 즐길 수 있는 휴대성을 자랑합니다.,It is a mini size and boasts portability that ...,It is a mini size and boasts portability that ...,8,14,1.750,None


In [69]:
TRAIN_DATA = train_df.to_dict("records")

TRAIN_DATA[0]

{'sn': 'INTSALDSUT062119042703238',
 'domain': '해외영업',
 'subdomain': '도소매유통',
 'ko_original': '원하시는 색상을 회신해 주시면 바로 제작 들어가겠습니다.',
 'ko': '원하시는 색상을 회신해 주시면 바로 제작 들어가겠습니다.',
 'mt': 'If you reply to the color you want, we will start making it right away.',
 'en': 'If you reply to the color you want, we will start making it right away.',
 'word_count_ko': 7,
 'word_count_en': 15,
 'word_ratio': 2.143,
 'ner': None}